In [5]:
import sys
sys.path.append('/data/user/wu_j1/bamf')

In [1]:
import pandas as pd
import numpy as np

In [7]:
from analysis_codes import helpers

The restrictions are in the form of 

$F[F_a]/F[F_b] \sim gaussian(0, width)$

F_a is essentially the index of the peak you want to restrict and
F_b is the index another peak you are using as a reference.

This means that for example you take mz 44 and mz 29 and say that in the first component of F 
mz 44 F_a = (0,loc(44))
divided by
mz 29 F_b = (0,loc(29))
is for example 0.4 +- some width.

Of course in practice F_a and F_b will be arrays of such restrictions, since you want multiple peaks restricted.

Width is basically how much you allow the peaks to vary, an arbitrary strength, I use 0.005 in the paper. We could assign the value inside the json file for running.

If you want to fully restrict a component that you already have, there is a helper function helpers.construct_restrictions
It assumes F is saved in pandas format as "F" in the file. 
NOTE: The G restrictions are untested and not currently discussed.

In [34]:
file = "/data/user/wu_j1/bamf/data/longer_data.h5"
res = helpers.construct_restrictions(F_file=file,G_file=None,F_to_restrict=["OOA", "HOA"], G_to_restrict=None,
                        reference_mz=29, reference_time=None)

In [35]:
res

{'restricted_components': array([b'A_DEC_W_037_HOA_t'], dtype='|S17'),
 'restriction_indices': [0],
 'F_a': array([[ 0,  0],
        [ 0,  1],
        [ 0,  2],
        [ 0,  3],
        [ 0,  4],
        [ 0,  6],
        [ 0,  7],
        [ 0,  8],
        [ 0,  9],
        [ 0, 10],
        [ 0, 11],
        [ 0, 12],
        [ 0, 13],
        [ 0, 14],
        [ 0, 15],
        [ 0, 16],
        [ 0, 17],
        [ 0, 18],
        [ 0, 19],
        [ 0, 20],
        [ 0, 21],
        [ 0, 22],
        [ 0, 23],
        [ 0, 24],
        [ 0, 25],
        [ 0, 26],
        [ 0, 27],
        [ 0, 28],
        [ 0, 29],
        [ 0, 30],
        [ 0, 31],
        [ 0, 32],
        [ 0, 33],
        [ 0, 34],
        [ 0, 35],
        [ 0, 36],
        [ 0, 37],
        [ 0, 38],
        [ 0, 39],
        [ 0, 40],
        [ 0, 41],
        [ 0, 42],
        [ 0, 43],
        [ 0, 44],
        [ 0, 45],
        [ 0, 46],
        [ 0, 47],
        [ 0, 48],
        [ 0, 49],
        [ 0

In [14]:
# This shows all the stuff that you need for the model
res.keys()

dict_keys(['restricted_components', 'restriction_indices', 'F_a', 'F_b', 'F_values'])

In [15]:
#these are just the names
res["restricted_components"]

array([b'OOA', b'HOA'], dtype='|S3')

In [17]:
#This says which components of F have restrictions (so even if only 1 pair of peaks in in component is, it must be in the list, because sorting doesn't touch them)
#I recommend always just going components from the start (so not for example [4,3]), since some assumptions have been made
res["restriction_indices"]

[0, 1]

In [22]:
#This is the indices (in F) of every peak except 29 (which is the F_b) in the first two components
print(res["F_a"].shape, res["F_a"][:2, :])

(146, 2) [[0 0]
 [0 1]]


In [33]:
#In this case we are comparing it to mz 29 of the same component
print(res["F_b"].shape, res["F_b"][:2, :])

(146, 2) [[0 8]
 [0 8]]


In [32]:
#Both components are in the same array
print(res["F_b"][int(res["F_b"].shape[0]/2), :])

[1 8]


In [38]:
#Then we have the peak ratios F[F_a]/F[F_b]
print(res["F_values"].shape, res["F_values"][:2])

(146,) [0.10816743 0.05037671]


In [39]:
#To store this in a format that the model will accept, we dump this as a normal hdf
#NB the weight is determined in the json, see for example ./json/camx_cons.json

helpers.dump_hdf5("data/example_full_restriction.h5", **res)

# smaller subset of restrictions
So say we wanted to build a smaller subset of restrictions, currently that would be manual

In [2]:
#We read in some F, for example let's take the same
F = pd.read_hdf("/data/user/wu_j1/bamf/data/longer_data.h5", "F")

In [3]:
F.loc[:, 43].to_numpy()[:, None]

array([[0.1103507 ],
       [0.12061194],
       [0.07013555]])

In [4]:
#We choose some reference mz (doesn't have to be the same for all pairs)
F.loc[:, 43]

OOA    0.110351
HOA    0.120612
COA    0.070136
Name: 43, dtype: float64

In [5]:
#We check what sort of values we get (These will be F_values)
F.loc["OOA", [43, 45, 57, 60]]/F.loc["OOA", 43]

43    1.000000
45    0.185436
57    0.156552
60    0.047026
Name: OOA, dtype: float64

In [23]:
#So now we intend to have 2 components (OOA, HOA) with 3 peaks (45,57,60) each referenced to mz 43
F_b = np.zeros([6,2], dtype=int)

In [24]:
#We fill the first column with [0,0,0,1,1,1], to indicate the components restricted
F_b[:3,0] = 0
F_b[3:,0] = 1
#Then fill the second column with the location of mz 43 
F_b[:,1] = F.columns.get_loc(43) 

In [25]:
F_b

array([[ 0, 17],
       [ 0, 17],
       [ 0, 17],
       [ 1, 17],
       [ 1, 17],
       [ 1, 17]])

In [17]:
#The amount of F_a must be the same as F_b
F_a = np.zeros([6,2], dtype=int)

In [18]:
#Same first column
F_a[:3,0] = 0
F_a[3:,0] = 1
#Locations of the other peaks for the second column
F_a[:3,1] = [F.columns.get_loc(x) for x in (45, 57, 60)]
F_a[3:,1] = [F.columns.get_loc(x) for x in (45, 57, 60)]

In [19]:
F_a

array([[ 0, 19],
       [ 0, 30],
       [ 0, 33],
       [ 1, 19],
       [ 1, 30],
       [ 1, 33]])

In [47]:
#Then we take all the ratios those indices indicate and concatenate them together to get F_values
F_values = np.concatenate((
(F.loc["OOA", [45, 57, 60]]/F.loc["OOA", 43]).to_numpy(), 
(F.loc["HOA", [45, 57, 60]]/F.loc["HOA", 43]).to_numpy()))

In [48]:
F_values

array([0.18543606, 0.15655182, 0.04702614, 0.01872373, 0.75024863,
       0.01979493])

In [49]:
#And we make a similar dictionary as the full constraints
smaller_constraints = {"F_a":F_a, "F_b":F_b, "F_values":F_values, "restricted_components":["OOA", "HOA"], "restriction_indices":[0,1]}

In [50]:
#same way to save it
helpers.dump_hdf5("./data/example_partial_restriction.h5", **smaller_constraints)